In [ ]:
# Malariagen API arguments for each call below.
# Replace a value to override it; keep the marker to use the call-site
# value or the method's default.
API_KEEP_CALLSITE_VALUE = '__KEEP_CALLSITE_VALUE__'
API_CALL_PARAMETERS = {'cell_4_call_1_af1_igv': {'region': '2RL:2,441,878-2,468,205', 'tracks': None, 'init': True},
 'cell_5_call_1_af1_view_alignments': {'region': '2RL:2,447,600-2,447,700',
                                       'sample': '__KEEP_CALLSITE_VALUE__',
                                       'visibility_window': 20000,
                                       'init': True,
                                       'snp_vcf_path': None},
 'cell_5_call_2_af1_sample_metadata': {'sample_sets': '1229-VO-GH-DADZIE-VMF00095',
                                       'sample_query': None,
                                       'sample_query_options': None,
                                       'sample_indices': None},
 'cell_7_call_1_af1_sample_metadata': {'sample_sets': None,
                                       'sample_query': None,
                                       'sample_query_options': None,
                                       'sample_indices': None},
 'cell_7_call_2_af1_snp_calls_to_vcf': {'output_path': '__KEEP_CALLSITE_VALUE__',
                                        'region': '__KEEP_CALLSITE_VALUE__',
                                        'sample_sets': '__KEEP_CALLSITE_VALUE__',
                                        'sample_query': '__KEEP_CALLSITE_VALUE__',
                                        'sample_query_options': None,
                                        'site_mask': None,
                                        'max_region_size': 1000000,
                                        'inline_array': True,
                                        'chunks': 'native',
                                        'overwrite': True,
                                        'fields': ('GT',)},
 'cell_10_call_1_af1_view_alignments': {'region': '__KEEP_CALLSITE_VALUE__',
                                        'sample': '__KEEP_CALLSITE_VALUE__',
                                        'visibility_window': 20000,
                                        'init': True,
                                        'snp_vcf_path': None}}


In [ ]:
import inspect
from memory_profiler import memory_usage


def _call_api_with_parameters(method, call_key, *args, **kwargs):
    signature = inspect.signature(method)
    bound = signature.bind_partial(*args, **kwargs)
    bound.apply_defaults()
    for name, value in API_CALL_PARAMETERS[call_key].items():
        if not (isinstance(value, str) and value == API_KEEP_CALLSITE_VALUE):
            bound.arguments[name] = value

    positional = []
    keyword = {}
    for parameter in signature.parameters.values():
        if parameter.kind is inspect.Parameter.POSITIONAL_ONLY:
            if parameter.name in bound.arguments:
                positional.append(bound.arguments[parameter.name])
        elif parameter.kind is inspect.Parameter.POSITIONAL_OR_KEYWORD:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_POSITIONAL:
            positional.extend(bound.arguments.get(parameter.name, ()))
        elif parameter.kind is inspect.Parameter.KEYWORD_ONLY:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_KEYWORD:
            keyword.update(bound.arguments.get(parameter.name, {}))
    return method(*positional, **keyword)


In [ ]:
import malariagen_data

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
%load_ext line_profiler
%load_ext memory_profiler

In [ ]:
af1 = malariagen_data.Af1()
af1

In [ ]:
_call_api_with_parameters(af1.igv, 'cell_4_call_1_af1_igv', region="2RL:2,441,878-2,468,205")

In [ ]:
_call_api_with_parameters(af1.view_alignments, 'cell_5_call_1_af1_view_alignments', sample=_call_api_with_parameters(af1.sample_metadata, 'cell_5_call_2_af1_sample_metadata', sample_sets="1229-VO-GH-DADZIE-VMF00095").iloc[0][
        "sample_id"
    ], region="2RL:2,447,600-2,447,700")

## Local tabix-indexed VCF in IGV
This exports one sample to a BGZF VCF and `.tbi` under the notebook working directory, then loads it as the SNP track. Both files must be inside the Jupyter server startup directory tree. The `snp_vcf_url` below assumes the server was started from the workspace root; if it was started from `notebooks/`, use the alternate URL shown in the code cell.

In [ ]:
from pathlib import Path

#%%capture cap --no-stderr

sample_set = "1240-VO-MZ-KOEKEMOER-VMF00101"
sample_id = str(
    _call_api_with_parameters(af1.sample_metadata, 'cell_7_call_1_af1_sample_metadata', sample_sets=sample_set).iloc[0]["sample_id"]
)
region = "2RL:2,440,000-2,445,000"

local_vcf_path = Path("local_igv_vcfs") / f"{sample_id}.vcf.gz"
local_vcf_path.parent.mkdir(parents=True, exist_ok=True)
regions = ["2RL:2,440,000-2,441,000", "2RL:2,440,000-2,442,000", "2RL:2,440,000-2,443,000", "2RL:2,440,000-2,444,000", 
                    "2RL:2,440,000-2,445,000", "2RL:2,440,000-2,446,000", "2RL:2,440,000-2,447,000", "2RL:2,440,000-2,448,000",
                    "2RL:2,440,000-2,449,000", "2RL:2,440,000-2,450,000"]
def run_snp_calls():
    _call_api_with_parameters(af1.snp_calls_to_vcf, 'cell_7_call_2_af1_snp_calls_to_vcf', output_path=str(local_vcf_path), region=region, sample_sets=[sample_set], sample_query=f"sample_id == '{sample_id}'", overwrite=True)


In [ ]:
#%%writefile snp_calls_to_vcf_5k_bp_time.txt
for region in regions:
    print(f"Time for region {region}")
    %timeit run_snp_calls()


In [ ]:
#%%writefile snp_calls_to_vcf_5k_bp_mem.txt 
for region in regions:
    print(f"Time for region {region}")
    %memit run_snp_calls()


In [ ]:
def view_alignments(region):
    _call_api_with_parameters(af1.view_alignments, 'cell_10_call_1_af1_view_alignments', sample=sample_id, region=region, snp_vcf_path=local_vcf_path)


In [ ]:
for region in regions:
    %timeit view_alignments(region)


In [ ]:
for region in regions:
    print(f"Memory usage for region {region}")
    %memit view_alignments(region)
